# 🔬 ForensiX: Colab Training & Benchmarking Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anas-rz/ForensiX/blob/main/notebooks/ForensiX_Colab_Train_and_Test.ipynb)

This notebook downloads real large datasets, extracts forensic features, trains, and evaluates **all ForensiX models** directly in Google Colab:

1. **`person-detector`**: Teeny tiny model (~23k params, LoRA PEFT) specialized for **one individual person** on CPU/GPU across image, audio, video, or audiovisual media.
2. **`id-motion`**: Temporal behavioural identity model (MediaPipe head-pose + blendshapes, supervised contrastive).
3. **`avcons`**: Audio-visual lip-sync consistency transformer (SyncNet embeddings).
4. **`avid`**: Cross-modal audio-visual identity (DINOv2 face + ECAPA-TDNN voice space).
5. **`physdf`**: Multimodal physics graph prior (masked autoencoder over face, pulse, lighting, and voice).

--- 
## 1. Setup & Environment
Clone the ForensiX repository and install dependencies.

In [ ]:
# Clone repository if not already present
!if [ ! -d "ForensiX" ]; then git clone https://github.com/anas-rz/ForensiX.git; fi
%cd ForensiX

# Install forensix with evaluation extras
!pip install -q -e .[eval]
!pip install -q soundfile safetensors open_clip_torch mediapipe

In [ ]:
import torch
import forensix

print(f"ForensiX version: {forensix.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:       {torch.cuda.get_device_name(0)}")

print("\nAvailable models:")
for m in forensix.list_models():
    print(f"  • {m}")

--- 
## 2. Mount Google Drive (Optional)
Mount Google Drive to automatically persist trained checkpoints (`.safetensors`) and benchmark reports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/forensix_checkpoints"
!mkdir -p {DRIVE_DIR}
print(f"Checkpoints will be backed up to: {DRIVE_DIR}")

--- 
## 3. One-Click Full Pipeline
Run the complete download, feature extraction, training, and benchmarking pipeline in a single command.

- Use `--preset minimal` for an ultra-fast verification run (< 3 minutes).
- Use `--preset standard` for full benchmark training with real dataset splits on GPU.

In [ ]:
# Execute full pipeline for all models
!python scripts/colab_train_and_test.py --all --preset minimal --drive

--- 
## 4. Train Teeny Tiny Specialized Person Model
Train a personalized deepfake detector specialized for **one individual** on CPU or GPU in seconds.
Supports photos, audio samples, or video clips.

In [ ]:
from forensix.specialized import train_person_model, PersonDetector

# 1. Provide genuine media of the person (photos, voice recordings, or videos)
real_photos = ["tests/data/face_public_domain.jpg", "tests/data/face2_public_domain.jpg"]
negative_photos = ["tests/data/other_face_public_domain.jpg"]

# 2. Train specialized model (< 25k params, CPU-friendly)
detector = train_person_model(
    person_name="neil_armstrong",
    real_media=real_photos,
    negative_media=negative_photos,
    modality="image",
    use_peft=True,
    epochs=30,
    lr=2e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# 3. Save checkpoint to .safetensors
detector.save("checkpoints/armstrong_detector.safetensors")
print("Trained specialized detector saved successfully!")

In [ ]:
# 4. Test prediction on genuine photo vs impostor / deepfake
pred_real = detector.predict("tests/data/face_public_domain.jpg")
print(f"Genuine query -> Fake Prob: {pred_real.fake_prob:.4f} | Label: {pred_real.label}")

pred_fake = detector.predict("tests/data/other_face_public_domain.jpg")
print(f"Impostor query -> Fake Prob: {pred_fake.fake_prob:.4f} | Label: {pred_fake.label}")

--- 
## 5. Train Multimodal ForensiX Zoo Models
Train and evaluate any of the zoo models (`id-motion`, `avcons`, `avid`, `physdf`) individually on real data.

In [ ]:
# Train id-motion (Behavioural Identity)
!python scripts/colab_train_and_test.py --train --models id-motion --steps 1000 --device cuda

In [ ]:
# Train avcons (Audio-Visual Lip-Sync Consistency)
!python scripts/colab_train_and_test.py --train --models avcons --steps 1000 --device cuda

In [ ]:
# Train avid (Audio-Visual Identity Heads & Scorer)
!python scripts/colab_train_and_test.py --train --models avid --steps 1000 --device cuda

In [ ]:
# Train physdf (Physics Graph Prior)
!python scripts/colab_train_and_test.py --train --models physdf --steps 1000 --device cuda

--- 
## 6. Run Benchmark & View Evaluation Report
Evaluate all checkpoints on held-out test splits and generate the final comparison table.

In [ ]:
!python scripts/colab_train_and_test.py --test --models all

In [ ]:
from IPython.display import Markdown, display

report_file = "checkpoints/evaluation_report.md"
try:
    with open(report_file, "r") as f:
        display(Markdown(f.read()))
except FileNotFoundError:
    print("Run the evaluation cell above to generate the report.")